In [3]:
import pandas as pd
import pyodbc

# 1. LOAD THE CLEANSED DATASET
df = pd.read_csv('../data/social_media_mental_health_clean.csv')
print(f"--- Standardised Dataset Loaded ---")
print(f"Total Rows to Process: {df.shape[0]}\n")

--- Standardised Dataset Loaded ---
Total Rows to Process: 1200



In [4]:
# =====================================================================
# 2. DIMENSION TABLE CREATION (Dim_Platform)
# =====================================================================
unique_platforms = df['platform_usage'].dropna().unique()
dim_platform = pd.DataFrame({
    'id_platform': range(1, len(unique_platforms) + 1),
    'platform_usage': unique_platforms
})

In [5]:
# =====================================================================
# 3. DIMENSION TABLE CREATION (Dim_Teenager)
# =====================================================================
dim_teenager = df[['age', 'gender', 'academic_performance', 'social_interaction_level']].copy()
dim_teenager.insert(0, 'id_teenager', range(1, len(dim_teenager) + 1))

In [6]:
# =====================================================================
# 4. FACT TABLE CREATION (Fact_Mental_Health_Impact)
# =====================================================================
fact_table = df.copy()
fact_table['id_teenager'] = dim_teenager['id_teenager']
fact_table = fact_table.merge(dim_platform, on='platform_usage', how='left')

fact_columns = [
    'id_teenager', 'id_platform', 'daily_social_media_hours', 
    'sleep_hours', 'screen_time_before_sleep', 'physical_activity', 
    'stress_level', 'anxiety_level', 'addiction_level', 'depression_label'
]
fact_mental_health_impact = fact_table[fact_columns].copy()
fact_mental_health_impact.insert(0, 'id_record', range(1, len(fact_mental_health_impact) + 1))

print("--- Dimensional Star Schema Generated Successfully ---\n")

--- Dimensional Star Schema Generated Successfully ---



In [7]:
# =====================================================================
# 5. PERSIST DATA INTO MICROSOFT SQL SERVER
# =====================================================================
# Define your SQL Server connection details
# Replace SERVER_NAME and DATABASE_NAME with your actual local configuration
server = 'localhost'
database = 'TeenMentalHealthDB'

# Connection string using Windows Authentication
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'

try:
    # Establish connection to SQL Server
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    
    # Enable fast execute many for high-performance batch inserts
    cursor.fast_executemany = True

    print("Connecting to SQL Server and creating tables...")

    # Drop tables if they already exist to avoid conflicts (Dumping staging style)
    cursor.execute("DROP TABLE IF EXISTS Fact_Mental_Health_Impact;")
    cursor.execute("DROP TABLE IF EXISTS Dim_Teenager;")
    cursor.execute("DROP TABLE IF EXISTS Dim_Platform;")

    # DDL: Create Dim_Platform
    cursor.execute("""
        CREATE TABLE Dim_Platform (
            id_platform INT PRIMARY KEY,
            platform_usage VARCHAR(100)
        );
    """)

    # DDL: Create Dim_Teenager
    cursor.execute("""
        CREATE TABLE Dim_Teenager (
            id_teenager INT PRIMARY KEY,
            age INT,
            gender VARCHAR(50),
            academic_performance VARCHAR(50),
            social_interaction_level VARCHAR(50)
        );
    """)

    # DDL: Create Fact_Mental_Health_Impact
    cursor.execute("""
        CREATE TABLE Fact_Mental_Health_Impact (
            id_record INT PRIMARY KEY,
            id_teenager INT FOREIGN KEY REFERENCES Dim_Teenager(id_teenager),
            id_platform INT FOREIGN KEY REFERENCES Dim_Platform(id_platform),
            daily_social_media_hours FLOAT,
            sleep_hours FLOAT,
            screen_time_before_sleep FLOAT,
            physical_activity FLOAT,
            stress_level INT,
            anxiety_level INT,
            addiction_level INT,
            depression_label VARCHAR(50)
        );
    """)
    conn.commit()

    # Insert Data into Dim_Platform
    print("Inserting data into Dim_Platform...")
    platform_rows = [tuple(x) for x in dim_platform.to_numpy()]
    cursor.executemany("INSERT INTO Dim_Platform (id_platform, platform_usage) VALUES (?, ?)", platform_rows)

    # Insert Data into Dim_Teenager
    print("Inserting data into Dim_Teenager...")
    teenager_rows = [tuple(x) for x in dim_teenager.to_numpy()]
    cursor.executemany("""
        INSERT INTO Dim_Teenager (id_teenager, age, gender, academic_performance, social_interaction_level) 
        VALUES (?, ?, ?, ?, ?)
    """, teenager_rows)

    # Insert Data into Fact_Mental_Health_Impact
    print("Inserting data into Fact_Mental_Health_Impact...")
    fact_rows = [tuple(x) for x in fact_mental_health_impact.to_numpy()]
    cursor.executemany("""
        INSERT INTO Fact_Mental_Health_Impact (
            id_record, id_teenager, id_platform, daily_social_media_hours, 
            sleep_hours, screen_time_before_sleep, physical_activity, 
            stress_level, anxiety_level, addiction_level, depression_label
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, fact_rows)

    # Commit all transactions
    conn.commit()
    print("\nSuccess! All data has been successfully modeled and loaded into SQL Server.")

except Exception as e:
    print(f"\nAn error occurred during SQL Server operations: {e}")
    if 'conn' in locals():
        conn.rollback()

finally:
    if 'conn' in locals():
        conn.close()

Connecting to SQL Server and creating tables...
Inserting data into Dim_Platform...
Inserting data into Dim_Teenager...
Inserting data into Fact_Mental_Health_Impact...

Success! All data has been successfully modeled and loaded into SQL Server.
